In [ ]:
# FULL MOVESET VERSION: 48 moves, 10 showcase routines, and 5 mixed combo routes.
FPS = 60  # Change to 30 if preferred.
REPEATS_PER_ROUTINE = 2  # Repeat each routine so every move is visible.

from itertools import cycle
from IPython.display import display
from PIL import Image
from street_fighter_common import make_sf2_env, unpack_step

SHOWCASE_ROUTINES = (
    ("movement_and_defense", ("walk_forward", "crouch", "walk_backward", "block_standing", "block_crouching")),
    ("standing_six_button_chain", ("standing_x", "standing_y", "standing_z", "standing_a", "standing_b", "standing_c")),
    ("crouching_six_button_chain", ("crouch_x", "crouch_y", "crouch_z", "crouch_a", "crouch_b", "crouch_c")),
    ("forward_jump_chain", ("jump_in_b", "jump_x", "jump_y", "jump_z", "jump_a", "jump_b", "jump_c")),
    ("neutral_jump_chain", ("neutral_jump", "neutral_jump_x", "neutral_jump_y", "neutral_jump_z", "neutral_jump_a", "neutral_jump_b", "neutral_jump_c")),
    ("back_jump_chain", ("back_jump_x", "back_jump_y", "back_jump_z", "back_jump_a", "back_jump_b", "back_jump_c")),
    ("triple_hadouken", ("hadouken_x", "hadouken_y", "hadouken_z")),
    ("triple_shoryuken", ("shoryuken_x", "shoryuken_y", "shoryuken_z")),
    ("triple_tatsumaki", ("tatsumaki_a", "tatsumaki_b", "tatsumaki_c")),
    ("throw_chain", ("throw_y", "throw_z")),
)

COMBO_ROUTES = (
    ("fireball_rush", ("hadouken_x", "walk_forward", "jump_in_b", "standing_x", "crouch_x")),
    ("dragon_punch_combo", ("jump_y", "standing_z", "crouch_x", "shoryuken_z")),
    ("hurricane_combo", ("jump_b", "standing_y", "crouch_b", "tatsumaki_c")),
    ("low_high_mix", ("crouch_a", "standing_x", "crouch_c", "jump_z", "hadouken_y")),
    ("throw_pressure", ("block_standing", "walk_forward", "standing_x", "throw_y", "shoryuken_x")),
)

WINNING_MIX = ("jump_in_b", "standing_x", "hadouken_x", "crouch_x", "standing_y", "hadouken_y")
env = make_sf2_env(action_mode="all", macro_actions=True, max_episode_steps=12000, shape_reward=False, render_mode="rgb_array")
frame_output = None
def show_frame(frame):
    global frame_output
    image = Image.fromarray(frame)
    if frame_output is None:
        frame_output = display(image, display_id=True)
    else:
        frame_output.update(image)
env.configure_realtime_render(FPS, frame_sink=show_frame)
action_by_name = {name: index for index, (name, _frames) in enumerate(env.macro_actions)}
all_showcase_moves = tuple(move for _routine, moves in SHOWCASE_ROUTINES for move in moves)
available_moves = {name for name in action_by_name if name != "idle"}
assert len(all_showcase_moves) == 48 and set(all_showcase_moves) == available_moves

def play_moves(label, moves, repeats=1):
    global done
    print(f"\n{label}: {', '.join(moves)}")
    for _ in range(repeats):
        for name in moves:
            if done:
                env.reset()
                done = False
            _obs, _reward, done, _info = unpack_step(env.step(action_by_name[name]))

try:
    env.reset()
    done = False
    print(f"FULL 48-MOVE SHOWCASE AT {FPS} FPS")
    for routine_name, moves in SHOWCASE_ROUTINES:
        play_moves(routine_name, moves, REPEATS_PER_ROUTINE)
    print("\nALL 48 MOVES USED. NOW PLAYING MIXED COMBINATIONS.")
    for combo_name, moves in COMBO_ROUTES:
        play_moves(combo_name, moves, 2)

    print("\nSTARTING FRESH VERIFIED WINNING MATCH.")
    env.reset()
    info = {}
    macro_steps = 0
    for name in cycle(WINNING_MIX):
        _obs, _reward, done, info = unpack_step(env.step(action_by_name[name]))
        macro_steps += 1
        if done:
            break
    print(f"result={info.get('terminal_reason')} Ryu={info.get('matches_won')} Guile={info.get('enemy_matches_won')} steps={macro_steps}")
finally:
    env.close()
